# E-FULLFT (final) — does the null depend on the adaptation method?

**Single self-contained notebook. Run all. ~5 h on the 96 GB runtime.**

Every result so far uses rank-16 LoRA on attention projections. A reviewer can reasonably ask
whether the null is a property of *LoRA* — a narrow subspace a fingerprint might simply have
nowhere to live in — rather than of the pipeline. Full fine-tuning frees every parameter. If device
identity still does not transfer, the null is a property of the **channel**.

### Two fixes baked in (both found the hard way)

**1 · fp32 master weights.** AdamW's step is ≈ LR regardless of gradient scale, so each update is
~2e-6. bf16 spacing at |θ| is |θ|/128, so updates survive **only** where |θ| < ~2.6e-4 — every
larger weight was frozen and the apparent drift came from the smallest weights alone. Measured:
bf16 gave relative drift ~0 at 22 GB; fp32 masters give 1.5e-04 at 44 GB.

**2 · exposure matched to LoRA.** LoRA saw 2000 × batch 2 = **4000 presentations / 1000 updates**.
`STEPS=4000` with `BATCH=1, GRAD_ACC=4` matches both. A null from an under-exposed run proves
nothing.

### Gates that halt the run
| stage | gate |
|---|---|
| F0 smoke | non-finite loss, or relative drift < 1e-5 (LR too low) |
| F1 | dead run, loss divergence over 400-step windows |
| **F1b** | **mean \|Δ\| vs base generations outside 15–90 / 255** — LoRA's reference is 31–59. Halts *before* the 2 h generation stage |
| F3 | incomplete measurement |

### Reading F4
| outcome | conclusion |
|---|---|
| null holds | the bound is a property of the channel, not of LoRA's rank or targets — **the strongest generality claim in the paper** |
| transfer appears | the null *was* a LoRA artefact; this changes the headline |
| generations degraded | **inconclusive** — report the degradation, do not report a bound |

In [ ]:
# %% 1 · CONFIG
import os, sys, json, glob, math, time, gc, shutil, hashlib, subprocess, random
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor
import csv as _csv
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"]=(userdata.get("HF_TOKEN") or "").strip()
except Exception: pass
from google.colab import drive; drive.mount("/content/drive", force_remount=False)
DRIVE="/content/drive/MyDrive"
BASE=os.path.join(DRIVE,"inv_channel/E_INV_P0_v3")
ROOT=os.path.join(DRIVE,"inv_channel/E_FULLFT")
for d in ["","ckpt","gens","csv","meta"]: os.makedirs(os.path.join(ROOT,d),exist_ok=True)
CACHE="/content/ft_cache"; os.makedirs(CACHE,exist_ok=True)

@dataclass
class CFG:
    STAGES: tuple = ("F0","F1","F1b","F2","F3","F4")   # full sequence
    MODEL: str = "stabilityai/stable-diffusion-3.5-medium"
    CROP: int = 1024
    ARMS: tuple = ("A","B"); SEEDS: tuple = (0,1,2)
    LR: float = 2e-6                 # full FT: 1e-6..5e-6, NOT LoRA's 1e-4
    STEPS: int = 4000                # 4000 x batch 1 x acc 4 == LoRA's 4000 presentations/1000 updates
    BATCH: int = 1; GRAD_ACC: int = 4
    CAPTION: str = "a photograph, sks style"
    G_PER: int = 500; GEN_SEED_BASE: int = 770000
    GEN_STEPS: int = 28; CFG_SCALE: float = 4.5; GEN_BATCH: int = 2
    GRAD_CKPT: bool = True
    MAX_LOSS_RATIO: float = 3.0
    MIN_REL_DRIFT: float = 1e-5      # F0 gate
    GATE_LO: float = 15.0; GATE_HI: float = 90.0      # F1b gate, gray levels vs base
    N_GATE: int = 16
    N_PROBE: int = 60
    N_PERM: int = 10000; ALPHA: float = 0.01
C=CFG(); MEAS=C.CROP
CFG_SHA=hashlib.sha256(json.dumps({k:v for k,v in asdict(C).items() if k!="STAGES"},
                                  sort_keys=True,default=str).encode()).hexdigest()
def stage(n):
    if n in C.STAGES: return True
    print(f"[{n}] SKIPPED"); return False
subprocess.run([sys.executable,"-m","pip","install","-q","PyWavelets","diffusers>=0.31",
                "transformers","accelerate","safetensors","sentencepiece","protobuf"],check=False)
import numpy as np, pandas as pd, pywt, torch
import torch.nn.functional as tF
from PIL import Image
from scipy import stats as sps
Image.MAX_IMAGE_PIXELS=None
DEV="cuda" if torch.cuda.is_available() else "cpu"
DT=torch.bfloat16 if DEV=="cuda" and torch.cuda.is_bf16_supported() else torch.float32
torch.backends.cudnn.allow_tf32=False; torch.backends.cuda.matmul.allow_tf32=False
np.seterr(divide="raise",invalid="raise"); _MD=DEV
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login; login(token=os.environ["HF_TOKEN"])
def HALT(m): raise SystemExit(f"\n{'='*70}\n[HALT] {m}\n{'='*70}")
if DEV!="cuda": HALT("this notebook needs a GPU")
vram=torch.cuda.get_device_properties(0).total_memory/2**30
print(f"[ENV] {torch.cuda.get_device_name(0)}  {vram:.0f} GB  compute dtype {DT}")
if vram<55: HALT(f"{vram:.0f} GB is not enough for fp32-master full FT (~45 GB peak). "
                 f"Use 8-bit Adam or a smaller model.")
def append_row(p,h,r):
    new=not os.path.exists(p)
    with open(p,"a") as f:
        if new: f.write(",".join(h)+"\n")
        f.write(",".join(str(x) for x in r)+"\n")
def done_keys(p,cols):
    if not os.path.exists(p): return set()
    with open(p) as f: return {tuple(r[k] for k in cols) for r in _csv.DictReader(f)}
def ajson(o,p,**kw):
    t=p+".tmp"; json.dump(o,open(t,"w"),**kw); os.replace(t,p)
def all_arms(): return [(f"{a}_ft_s{s}",a,s) for s in C.SEEDS for a in C.ARMS]
def all_tags(): return [t for t,_,_ in all_arms()]
def cdir(t): return os.path.join(ROOT,"ckpt",t)
def gdir(t): return os.path.join(ROOT,"gens",t)
def tdir(role): return os.path.join(BASE,"train_png",f"{role}_raw")
KFP={r:np.load(os.path.join(BASE,"fingerprints",f"K_{r}_E2.npy")) for r in ("A","B")}
for r in C.ARMS:
    n=len(glob.glob(os.path.join(tdir(r),"*.png")))
    if n<10: HALT(f"training PNGs missing for {r}: {tdir(r)}")
print(f"[CFG] SHA {CFG_SHA[:16]} | {len(all_arms())} runs | LR {C.LR} | STEPS {C.STEPS} "
      f"(= {C.STEPS*C.BATCH} presentations, {C.STEPS//C.GRAD_ACC} updates; LoRA was 4000/1000)")

In [ ]:
# %% 2 · forensic core + prompt cache (measurement path identical to every other experiment)
from diffusers import StableDiffusion3Pipeline
def _cp(fp): return os.path.join(CACHE,hashlib.sha1(f"{fp}|{MEAS}".encode()).hexdigest()[:16]+".npy")
def load_lum(fp,cache=True):
    if cache and os.path.exists(_cp(fp)): return np.load(_cp(fp))
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        a=np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
    l=(0.299*a[...,0]+0.587*a[...,1]+0.114*a[...,2]).astype(np.float32)
    if cache: np.save(_cp(fp),l)
    return l
def _c2(x,k):
    xt=torch.from_numpy(np.ascontiguousarray(x))[None,None]; kt=torch.from_numpy(k)[None,None]
    if _MD=="cuda": return tF.conv2d(xt.cuda(),kt.cuda(),padding=k.shape[0]//2)[0,0].cpu().numpy()
    return tF.conv2d(xt,kt,padding=k.shape[0]//2)[0,0].numpy()
def wres(img):
    s0=np.float32(4.0)
    co=pywt.wavedec2(img,"db8",level=4,mode="periodization"); out=[co[0]*0.0]
    for (cH,cV,cD) in co[1:]:
        bnd=[]
        for ch in (cH,cV,cD):
            ch=ch.astype(np.float32); vm=None
            for w in (3,5,7,9):
                v=np.maximum(_c2(ch*ch,np.ones((w,w),np.float32)/(w*w))-s0,0.0)
                vm=v if vm is None else np.minimum(vm,v)
            bnd.append(ch*(s0/(vm+s0)))
        out.append(tuple(bnd))
    n=pywt.waverec2(out,"db8",mode="periodization")[:img.shape[0],:img.shape[1]]
    r=n-n.mean(); sd=float(r.std())
    if not np.isfinite(sd) or sd==0: HALT("degenerate residual")
    return (r/sd).astype(np.float32)
def nccb(a,B):
    a=a-a.mean(); na=a.norm()
    Bc=B-B.mean(dim=(1,2),keepdim=True); nb=Bc.flatten(1).norm(dim=1)
    return (Bc*a).flatten(1).sum(dim=1)/(na*nb).clamp_min(1e-30)
def load_base(with_text=False):
    kw=dict(torch_dtype=DT, token=True)
    if not with_text: kw.update(text_encoder=None,text_encoder_2=None,text_encoder_3=None,
                                tokenizer=None,tokenizer_2=None,tokenizer_3=None)
    try:    return StableDiffusion3Pipeline.from_pretrained(C.MODEL,**kw).to(DEV)
    except Exception:
        p=StableDiffusion3Pipeline.from_pretrained(C.MODEL,torch_dtype=DT,token=True).to(DEV)
        p.text_encoder=p.text_encoder_2=p.text_encoder_3=None; return p
def cached_prompt():
    p=os.path.join(ROOT,"ckpt","_prompt.pt")
    if os.path.exists(p):
        z=torch.load(p,map_location=DEV); return z["pe"].to(DEV),z["pp"].to(DEV)
    pi=StableDiffusion3Pipeline.from_pretrained(C.MODEL,torch_dtype=DT,token=True).to(DEV)
    with torch.no_grad():
        pe,_,pp,_=pi.encode_prompt(prompt=C.CAPTION,prompt_2=C.CAPTION,prompt_3=C.CAPTION,
                                   device=DEV,num_images_per_prompt=1,
                                   do_classifier_free_guidance=False)
    torch.save({"pe":pe.cpu(),"pp":pp.cpu()},p); del pi
    gc.collect(); torch.cuda.empty_cache()
    return pe.to(DEV),pp.to(DEV)
print("[core] ready")

In [ ]:
# %% 3 · train_full — fp32 MASTER WEIGHTS, bf16 compute, RELATIVE drift.  + F0 smoke
def train_full(tag, role, seed, steps, pe, pp, save=True):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    pipe=load_base(); vae,tr,sch = pipe.vae, pipe.transformer, pipe.scheduler
    vae.requires_grad_(False)
    tr = tr.float(); tr.requires_grad_(True)          # <-- fp32 masters; bf16 loses a ~2e-6 step
    if C.GRAD_CKPT: tr.enable_gradient_checkpointing()
    tr.train()
    names=[n for n,_ in tr.named_parameters()]
    probe=[names[i] for i in np.linspace(0,len(names)-1,min(C.N_PROBE,len(names))).astype(int)]
    P=dict(tr.named_parameters())
    theta0={k:P[k].detach().clone() for k in probe}
    norm0=float(np.sqrt(sum(float((v**2).sum()) for v in theta0.values())))
    ps=[p for p in tr.parameters() if p.requires_grad]
    print(f"[FT] {tag}: {sum(p.numel() for p in ps)/1e9:.2f}B trainable (fp32 masters), LR {C.LR}")
    opt=torch.optim.AdamW(ps, lr=C.LR, weight_decay=0.0); opt.zero_grad(set_to_none=True)
    files=sorted(glob.glob(os.path.join(tdir(role),"*.png")))
    cache=torch.stack([torch.from_numpy(
        np.asarray(Image.open(f).convert("RGB"),np.float32).copy()/127.5-1.0).permute(2,0,1)
        for f in files])
    sf=float(vae.config.scaling_factor); shf=float(vae.config.shift_factor or 0.0)
    ts,sg=sch.timesteps.to(DEV), sch.sigmas.to(DEV); nts=int(sch.config.num_train_timesteps)
    if len(sg)!=nts or len(ts)!=nts: HALT("scheduler array length mismatch — pin diffusers")
    g=torch.Generator().manual_seed(seed); losses=[]; t0=time.time()
    W=max(20, steps//10)
    for st in range(steps):
        idx=torch.randint(0,cache.shape[0],(C.BATCH,),generator=g)
        px=cache[idx].to(DEV, dtype=DT)
        with torch.no_grad():
            lat=vae.encode(px).latent_dist.sample(); lat=((lat-shf)*sf).float()
        noise=torch.randn_like(lat)
        u=torch.sigmoid(torch.randn(lat.shape[0],device=DEV,dtype=torch.float32))
        ii=(u*nts).long().clamp_(0,nts-1); sig=sg[ii].float().view(-1,1,1,1)
        noisy=(1.0-sig)*lat + sig*noise
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            pred=tr(hidden_states=noisy, timestep=ts[ii],
                    encoder_hidden_states=pe.repeat(lat.shape[0],1,1).float(),
                    pooled_projections=pp.repeat(lat.shape[0],1).float(),
                    return_dict=False)[0]
        loss=tF.mse_loss(pred.float(), (noise-lat).float())/C.GRAD_ACC
        loss.backward()
        if (st+1)%C.GRAD_ACC==0:
            torch.nn.utils.clip_grad_norm_(ps,1.0); opt.step(); opt.zero_grad(set_to_none=True)
        losses.append(float(loss.detach())*C.GRAD_ACC)
        if not np.isfinite(losses[-1]): HALT(f"{tag}: non-finite loss at step {st+1}")
        if (st+1)%max(20,steps//10)==0:
            Pn=dict(tr.named_parameters())
            dn=float(np.sqrt(sum(float(((Pn[k].detach()-v)**2).sum()) for k,v in theta0.items())))
            el=time.time()-t0
            print(f"    {st+1}/{steps} loss {np.mean(losses[-W:]):.5f} rel-drift {dn/norm0:.3e} "
                  f"peak {torch.cuda.max_memory_allocated()/2**30:.0f}GB "
                  f"eta {(el/(st+1))*(steps-st-1)/60:.1f}min")
    open_l=float(np.mean(losses[:W])); tail_l=float(np.mean(losses[-W:]))
    if tail_l > C.MAX_LOSS_RATIO*open_l:
        HALT(f"{tag}: loss diverged ({open_l:.4f} -> {tail_l:.4f}). Lower LR.")
    Pn=dict(tr.named_parameters())
    dnorm=float(np.sqrt(sum(float(((Pn[k].detach()-v)**2).sum()) for k,v in theta0.items())))
    rel=dnorm/norm0
    print(f"[FT] {tag}: loss {open_l:.5f} -> {tail_l:.5f} | RELATIVE drift {rel:.4e} "
          f"| {(time.time()-t0)/60:.1f} min")
    if save:
        os.makedirs(cdir(tag),exist_ok=True)
        tr.to(torch.bfloat16).save_pretrained(os.path.join(cdir(tag),"transformer"))
        ajson({"tag":tag,"role":role,"seed":seed,"lr":C.LR,"steps":steps,"config_sha":CFG_SHA,
               "open_loss":open_l,"tail_loss":tail_l,"drift_relative":rel,"probe_norm":norm0,
               "precision":"fp32_master_bf16_compute","minutes":(time.time()-t0)/60,
               "effective":True}, os.path.join(cdir(tag),"train_meta.json"),indent=2)
    del pipe,tr,vae,cache,theta0; gc.collect(); torch.cuda.empty_cache()
    return open_l, tail_l, rel

if stage("F0"):
    pe,pp=cached_prompt()
    _,_,rel=train_full("SMOKE","A",0,30,pe,pp,save=False)
    print(f"\n[F0] peak {torch.cuda.max_memory_allocated()/2**30:.0f} GB | rel drift {rel:.3e}")
    if rel < C.MIN_REL_DRIFT:
        HALT(f"relative drift {rel:.2e} < {C.MIN_REL_DRIFT:.0e} — the model is not moving. "
             f"Raise C.LR (try 1e-5) and re-run.")
    print("[F0] PASS")

In [ ]:
# %% 4 · F1 — six full fine-tuning runs
def ft_valid(tag):
    p=os.path.join(cdir(tag),"train_meta.json")
    if not os.path.isdir(os.path.join(cdir(tag),"transformer")): return False
    m=json.load(open(p)) if os.path.exists(p) else None
    return (m is not None and m.get("config_sha")==CFG_SHA and m.get("effective",False)
            and m.get("precision")=="fp32_master_bf16_compute")
if stage("F1"):
    free=shutil.disk_usage(ROOT).free/2**30
    print(f"[F1] free at ROOT {free:.0f} GB | need ~4.5 GB x {len(all_arms())} "
          f"= {4.5*len(all_arms()):.0f} GB")
    if free < 4.5*len(all_arms())+10:
        print("[F1] TIGHT — uncomment the rmtree in F2 to delete each checkpoint after generation.")
    pe,pp=cached_prompt()
    for i,(tag,role,seed) in enumerate(all_arms(),1):
        if ft_valid(tag): print(f"[F1] {i}/{len(all_arms())} skip valid {tag}"); continue
        print(f"\n[F1] {i}/{len(all_arms())}: {tag}")
        train_full(tag,role,seed,C.STEPS,pe,pp,save=True)
    bad=[t for t in all_tags() if not ft_valid(t)]
    if bad: HALT(f"F1 incomplete: {bad}")
    print(f"\n[F1] PASS\n{'run':14s} {'open':>9s} {'tail':>9s} {'rel drift':>11s} {'min':>6s}")
    _r=[]
    for t in all_tags():
        m=json.load(open(os.path.join(cdir(t),"train_meta.json"))); _r.append(m["drift_relative"])
        print(f"{t:14s} {m['open_loss']:9.5f} {m['tail_loss']:9.5f} "
              f"{m['drift_relative']:11.3e} {m['minutes']:6.1f}")
    cv=100*np.std(_r,ddof=1)/np.mean(_r)
    print(f"\n  relative drift: mean {np.mean(_r):.3e}  CV {cv:.0f}%")
    if cv>50: print("  *** CV > 50%: the runs are NOT comparable; the adapter-level test is "
                    "weakened. Report this. ***")

In [ ]:
# %% 5 · F1b — ADAPTATION-STRENGTH GATE. Halts before the 2 h generation stage.
if stage("F1b"):
    from diffusers import SD3Transformer2DModel
    BASE_GENS=os.path.join(BASE,"gens","base")
    if not os.path.isdir(BASE_GENS): HALT(f"base generations not found: {BASE_GENS}")
    bank=[(C.CAPTION,C.GEN_SEED_BASE+j) for j in range(C.G_PER)]
    rows=[]
    for tag in all_tags():
        tr=SD3Transformer2DModel.from_pretrained(os.path.join(cdir(tag),"transformer"),
                                                 torch_dtype=DT)
        pipe=StableDiffusion3Pipeline.from_pretrained(C.MODEL,transformer=tr,torch_dtype=DT,
                                                      token=True).to(DEV)
        pipe.set_progress_bar_config(disable=True); d=[]
        try:
            for s in range(0,C.N_GATE,C.GEN_BATCH):
                ix=list(range(s,min(s+C.GEN_BATCH,C.N_GATE)))
                gs=[torch.Generator(device=DEV).manual_seed(bank[i][1]) for i in ix]
                with torch.inference_mode():
                    imgs=pipe(prompt=[bank[i][0] for i in ix],num_inference_steps=C.GEN_STEPS,
                              guidance_scale=C.CFG_SCALE,height=MEAS,width=MEAS,
                              generator=gs).images
                for i,im in zip(ix,imgs):
                    bp=os.path.join(BASE_GENS,f"{i:05d}.png")
                    if not os.path.exists(bp): continue
                    a=np.asarray(im.convert("RGB"),np.float32)
                    with Image.open(bp) as b_:
                        b_=b_.convert("RGB"); W_,H_=b_.size
                        b=np.asarray(b_.crop(((W_-MEAS)//2,(H_-MEAS)//2,(W_-MEAS)//2+MEAS,
                                              (H_-MEAS)//2+MEAS)),np.float32)
                    d.append(float(np.abs(a-b).mean()))
        finally:
            del pipe,tr; gc.collect(); torch.cuda.empty_cache()
        rows.append((tag,float(np.mean(d)),float(np.std(d))))
        print(f"[F1b] {tag:14s} mean |delta| vs base = {np.mean(d):6.2f} / 255 (sd {np.std(d):.2f})")
    m=np.array([r[1] for r in rows])
    ajson({"per_run":rows,"lora_reference":[31,59],"gate":[C.GATE_LO,C.GATE_HI]},
          os.path.join(ROOT,"F1b_adaptation_gate.json"),indent=2)
    print("\n"+"="*72)
    print(f"  full FT: {m.min():.1f} - {m.max():.1f} / 255      LoRA reference: 31 - 59 / 255")
    if m.max() < C.GATE_LO:
        HALT(f"BARELY ADAPTED (max {m.max():.1f} < {C.GATE_LO}). A null from these runs says "
             f"nothing about full fine-tuning. Raise C.LR (try 1e-5) or C.STEPS and re-run F1.")
    if m.min() > C.GATE_HI:
        HALT(f"LIKELY DEGRADED (min {m.min():.1f} > {C.GATE_HI}). Inspect generations before "
             f"proceeding; a bound from broken generations is worse than no bound.")
    print("  GATE PASSED — adaptation is comparable to LoRA. Proceeding to F2.")
    print("="*72)

In [ ]:
# %% 6 · F2 — generation from each fine-tuned transformer (same paired seed bank)
if stage("F2"):
    from diffusers import SD3Transformer2DModel
    bank=[(C.CAPTION,C.GEN_SEED_BASE+j) for j in range(C.G_PER)]
    for tag in all_tags():
        d_=gdir(tag); os.makedirs(d_,exist_ok=True)
        for t_ in glob.glob(os.path.join(d_,"*.tmp")): os.remove(t_)
        miss=[i for i in range(C.G_PER) if not os.path.exists(os.path.join(d_,f"{i:05d}.png"))]
        if not miss: print(f"[F2] skip complete {tag}"); continue
        tr=SD3Transformer2DModel.from_pretrained(os.path.join(cdir(tag),"transformer"),
                                                 torch_dtype=DT)
        pipe=StableDiffusion3Pipeline.from_pretrained(C.MODEL,transformer=tr,torch_dtype=DT,
                                                      token=True).to(DEV)
        pipe.set_progress_bar_config(disable=True)
        print(f"[F2] {tag}: {len(miss)} missing"); t0=time.time()
        try:
            for s in range(0,len(miss),C.GEN_BATCH):
                ix=miss[s:s+C.GEN_BATCH]
                gs=[torch.Generator(device=DEV).manual_seed(bank[i][1]) for i in ix]
                with torch.inference_mode():
                    imgs=pipe(prompt=[bank[i][0] for i in ix],num_inference_steps=C.GEN_STEPS,
                              guidance_scale=C.CFG_SCALE,height=MEAS,width=MEAS,
                              generator=gs).images
                for i,im in zip(ix,imgs):
                    t_=os.path.join(d_,f"{i:05d}.png.tmp")
                    im.save(t_,format="PNG"); os.replace(t_,os.path.join(d_,f"{i:05d}.png"))
                if (s//C.GEN_BATCH)%50==0: print(f"    {s+len(ix)}/{len(miss)}")
        finally:
            del pipe,tr; gc.collect(); torch.cuda.empty_cache()
        print(f"[F2] complete {tag} ({(time.time()-t0)/60:.0f} min)")
        # DISK: uncomment to free 4.5 GB once this tag's generations exist
        # shutil.rmtree(os.path.join(cdir(tag),"transformer"), ignore_errors=True)
    print("[F2] PASS")

In [ ]:
# %% 7 · F3 — measurement, identical statistic to every other experiment
if stage("F3"):
    K=torch.stack([torch.from_numpy(KFP[r]) for r in ("A","B")]).to(_MD)
    p=os.path.join(ROOT,"csv","f3_measure.csv"); done=done_keys(p,["tag","gen_idx","K"])
    ex=ThreadPoolExecutor(max_workers=4)
    try:
        for tag in all_tags():
            paths=[os.path.join(gdir(tag),f"{i:05d}.png") for i in range(C.G_PER)]
            todo=[i for i in range(C.G_PER)
                  if not all((tag,str(i),r) in done for r in ("A","B"))]
            if not todo: print(f"[F3] {tag}: complete"); continue
            fut={i:ex.submit(load_lum,paths[i],False) for i in todo[:6]}; t0=time.time()
            for pos,i in enumerate(todo):
                Z_np=fut.pop(i).result(); nx=pos+6
                if nx<len(todo): fut[todo[nx]]=ex.submit(load_lum,paths[todo[nx]],False)
                W=torch.from_numpy(wres(Z_np)).to(_MD); Z=torch.from_numpy(Z_np).to(_MD)
                v=nccb(W,Z.unsqueeze(0)*K).cpu().numpy()
                for j,r in enumerate(("A","B")):
                    if (tag,str(i),r) in done: continue
                    append_row(p,["tag","gen_idx","K","rho"],[tag,i,r,f"{v[j]:.6e}"])
            print(f"[F3] {tag}: {len(todo)} measured ({time.time()-t0:.0f}s)")
    finally:
        ex.shutdown(wait=True); gc.collect(); torch.cuda.empty_cache()
    n=len(done_keys(p,["tag","gen_idx","K"])); exp=len(all_tags())*C.G_PER*2
    if n!=exp: HALT(f"F3 incomplete: {n}/{exp}")
    print(f"[F3] PASS — {n}/{exp} rows")

In [ ]:
# %% 8 · F4 — adapter-level IUT + the SYMMETRIC statistic (additive term cancels)
if stage("F4"):
    rng=np.random.default_rng(0)
    d=pd.read_csv(os.path.join(ROOT,"csv","f3_measure.csv"))
    def th(tag,own,oth):
        a=d[(d.tag==tag)&(d.K==own)].sort_values("gen_idx")["rho"].to_numpy(float)
        b=d[(d.tag==tag)&(d.K==oth)].sort_values("gen_idx")["rho"].to_numpy(float)
        n=min(len(a),len(b)); return a[:n]-b[:n]
    def sf(x,n=C.N_PERM):
        o=x.mean(); c=sum((x*rng.choice((-1.,1.),len(x))).mean()>=o for _ in range(n))
        return (c+1)/(n+1)
    out={"config_sha":CFG_SHA,"arms":{}}; per={}
    print(f"{'arm':4s} {'run':14s} {'theta':>12s} {'tail loss':>10s} {'rel drift':>11s}")
    for arm,own,oth in (("A","A","B"),("B","B","A")):
        v=[]
        for s in C.SEEDS:
            t=f"{arm}_ft_s{s}"; x=th(t,own,oth)
            if not len(x): continue
            v.append(x); m=json.load(open(os.path.join(cdir(t),"train_meta.json")))
            print(f"{arm:4s} {t:14s} {x.mean():+12.3e} {m['tail_loss']:10.5f} "
                  f"{m['drift_relative']:11.3e}")
        mm=np.array([x.mean() for x in v]); per[arm]=mm; k=len(mm)
        tc=float(sps.t.ppf(1-C.ALPHA/2,df=k-1)); U=float(mm.mean()+tc*mm.std(ddof=1)/np.sqrt(k))
        out["arms"][arm]={"n":k,"means":mm.tolist(),"theta":float(mm.mean()),
                          "sd":float(mm.std(ddof=1)),"U":U,"p_adapter":sf(mm)}
        print(f"     -> n={k} theta={mm.mean():+.3e} sd={mm.std(ddof=1):.3e} U={U:+.3e} "
              f"p={out['arms'][arm]['p_adapter']:.4g}\n")
    sym=(per["A"]+per["B"])/2; k=len(sym)
    tc=float(sps.t.ppf(1-C.ALPHA/2,df=k-1)); Usym=float(sym.mean()+tc*sym.std(ddof=1)/np.sqrt(k))
    R_real=3.56703e-02
    out["symmetric"]={"per_seed":sym.tolist(),"theta_sym":float(sym.mean()),
                      "sd":float(sym.std(ddof=1)),"U":Usym,"lambda_U":Usym/R_real,
                      "t":float(sym.mean()/(sym.std(ddof=1)/np.sqrt(k)))}
    IUT=all(out["arms"][a]["p_adapter"]<C.ALPHA for a in ("A","B")); out["IUT_pass"]=bool(IUT)
    print("="*72)
    print("FULL FINE-TUNING — IUT: "
          + ("PASS — the null WAS a LoRA artefact" if IUT
             else "FAIL — the null HOLDS under full fine-tuning"))
    print(f"  theta_sym = {sym.mean():+.3e}   t = {out['symmetric']['t']:+.2f}")
    print(f"  lambda_U (symmetric, n={k}) = {Usym/R_real:.4%}   [FLAG AS n=3, full FT]")
    print("="*72)
    print("BEFORE REPORTING: run the base study's S4 copy audit on these generations.")
    print("LoRA gave 0.0%. Full FT on 50 images overfits; a high copy rate means memorisation is")
    print("doing the work and the device statistic is not interpretable. Eyeball ~20 per arm too.")
    ajson(out,os.path.join(ROOT,"E_FULLFT_results.json"),indent=2)

## Run all

`STAGES` is already the full sequence. **Runtime → Run all**; approve the Drive mount and walk away.
Every stage is resumable and skip-if-complete, so if the session drops just Run all again.

| stage | wall | halts on |
|---|---|---|
| F0 smoke | ~1 min | non-finite loss, or rel drift < 1e-5 |
| F1 | ~2.7 h | dead run, divergence |
| **F1b** | ~4 min | **adaptation outside 15–90 / 255** |
| F2 | ~2 h | — |
| F3 | ~15 min | incomplete measurement |
| F4 | ~5 min | — |

**Expect `IUT: FAIL`.** That is the desired outcome: it means the null holds when every parameter is
free to move, so the bound is a property of the channel rather than of LoRA's rank or target set.
Report the bound flagged **n = 3, full fine-tuning** — t₀.₉₉₅,₂ = 9.92 makes it far looser than the
n = 6 primary, and the two are not comparable.

**Do not run the old E_FULLFT notebook after this one.** It defines `train_full` in bf16 and would
silently undo the fp32-master fix.